# Preparación para Modelado — Siniestros Fatales ONSV 2021–2025
**Curso:** Agentes Inteligentes | **Grupo 5**

Este notebook toma el dataset limpio (`siniestros_limpio.csv`) producido en el notebook 01
y lo prepara para el modelado supervisado. En el notebook 01 ya se resolvieron los
problemas de calidad del dato (nulos, tipos, encoding). Este notebook se ocupa
exclusivamente de decisiones de modelado (Géron, 2022).

1. Carga y verificación del dataset limpio
2. Selección de features con justificación por columna
3. Encoding de variables categóricas
4. Verificación del desbalance de clases
5. Aplicación de SMOTE
6. Exportación del dataset listo para modelado

## 1. Carga y verificación

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter

plt.rcParams['figure.figsize'] = (10, 4)
plt.rcParams['font.size'] = 11
sns.set_style('whitegrid')

df = pd.read_csv('siniestros_limpio.csv', encoding='utf-8')

print(f'Dimensiones cargadas: {df.shape[0]:,} filas × {df.shape[1]} columnas')
print('\nTipos de datos detectados:')
print(df.dtypes)
print(f'\nNulos restantes: {df.isnull().sum().sum()}')

## 2. Selección de features

**Criterio general (Provost & Fawcett, 2013):** se excluyen variables con
redundancia semántica, cardinalidad excesiva sin poder predictivo diferencial,
o que son derivaciones directas de la variable objetivo.

| Columna | Decisión | Razón |
|---|---|---|
| `CANTIDAD DE FALLECIDOS` | **Excluir** | Duplicada en `FALLECIDOS` (variable transformada) |
| `CANTIDAD DE LESIONADOS` | **Excluir** | Duplicada en `LESIONADOS` |
| `MORTALIDAD_MULTIPLE` | **Excluir del X** | Es parte de la variable objetivo, causa data leakage |
| `LESION_MULTIPLE` | **Excluir del X** | Ídem |
| `FALLECIDOS` | **Excluir del X** | Construye directamente la variable objetivo |
| `LESIONADOS` | **Excluir del X** | Ídem |
| `COD CARRETERA` | **Excluir** | 1,045 valores únicos; redundante con `RED VIAL` (5 categorías) |
| `PROVINCIA` | **Excluir** | 182 valores únicos; capturada por `DEPARTAMENTO` + `REGION_NATURAL` |
| `DISTRITO` | **Excluir** | 1,086 valores únicos; información espacial cubierta por coordenadas |
| `CLASIFICACIÓN SEÑAL Nº 1` | **Excluir** | 85-94% SIN_DATO; información ya capturada en `¿EXISTE SEÑAL?` |
| `CLASIFICACIÓN SEÑAL Nº 2` | **Excluir** | Ídem |
| `CAUSA ESPECÍFICA` | **Excluir** | 38 categorías; altamente correlada con `CAUSA FACTOR PRINCIPAL` |
| `CATEGORIA_SEVERIDAD` | **Target (y)** | Variable objetivo del modelo |
| Resto | **Incluir** | Variables predictoras con cardinalidad manejable |

In [ ]:
# Columnas a excluir — detectadas dinámicamente por nombre parcial
EXCLUIR = [
    'CANTIDAD DE FALLECIDOS', 'CANTIDAD DE LESIONADOS',
    'MORTALIDAD_MULTIPLE', 'LESION_MULTIPLE',
    'FALLECIDOS', 'LESIONADOS',
    'COD CARRETERA', 'PROVINCIA', 'DISTRITO',
    'CAUSA ESPECÍFICA',
]

# Excluir columnas de clasificación de señal (cardinalidad baja pero 85-94% SIN_DATO)
excluir_señal = [c for c in df.columns if 'CLASIFICACIÓN' in c.upper()]
EXCLUIR += excluir_señal

# Variable objetivo
TARGET = 'CATEGORIA_SEVERIDAD'

# Verificar que todas las columnas a excluir existen
excluir_real = [c for c in EXCLUIR if c in df.columns]
no_encontradas = [c for c in EXCLUIR if c not in df.columns]
if no_encontradas:
    print(f'⚠ No encontradas (ya excluidas o nombre distinto): {no_encontradas}')

# Construir X e y
X = df.drop(columns=excluir_real + [TARGET], errors='ignore')
y = df[TARGET]

print(f'Features seleccionadas ({X.shape[1]}):')
for col in X.columns:
    print(f'  {col} — {X[col].dtype} — {X[col].nunique()} únicos')

## 3. Encoding de variables categóricas

Random Forest no acepta strings directamente. Se usa **Label Encoding** para
variables ordinales y **One-Hot Encoding** para nominales con pocas categorías.

Para variables con muchas categorías (como `DEPARTAMENTO` con 25) se usa
Label Encoding — Random Forest maneja bien esta representación al dividir
por umbral numérico en cada nodo (Breiman, 2001).

In [ ]:
from sklearn.preprocessing import LabelEncoder

# Separar columnas por tipo
cols_cat = X.select_dtypes(include='object').columns.tolist()
cols_num = X.select_dtypes(include=np.number).columns.tolist()

print(f'Columnas categóricas a encodear ({len(cols_cat)}): {cols_cat}')
print(f'Columnas numéricas (sin cambio) ({len(cols_num)}): {cols_num}')

In [ ]:
# Aplicar Label Encoding dinámicamente
# Se guarda el encoder de cada columna para poder invertir después si se necesita
X_enc = X.copy()
encoders = {}

for col in cols_cat:
    le = LabelEncoder()
    X_enc[col] = le.fit_transform(X_enc[col].astype(str))
    encoders[col] = le
    print(f'  {col}: {len(le.classes_)} categorías → encoded')

# Encodear variable objetivo
le_target = LabelEncoder()
y_enc = le_target.fit_transform(y)
print(f'\nClases en variable objetivo: {list(le_target.classes_)}')
print(f'Encodeo: {dict(zip(le_target.classes_, le_target.transform(le_target.classes_)))}')

In [ ]:
# Verificación final antes de SMOTE
print('=== Dataset listo para SMOTE ===')
print(f'X shape: {X_enc.shape}')
print(f'y shape: {y_enc.shape}')
print(f'Nulos en X: {X_enc.isnull().sum().sum()}')
print(f'\nDistribución de clases (antes de SMOTE):')
conteo = Counter(y_enc)
for clase, n in sorted(conteo.items()):
    nombre = le_target.classes_[clase]
    print(f'  {clase} ({nombre}): {n:,} ({n/len(y_enc)*100:.1f}%)')

## 4. Aplicación de SMOTE

**Justificación (Bazarnovi & Mohammadian, 2024):** el desbalance extremo de clases
(84% LEVE vs 10% y 6% para las clases graves) sesga el modelo hacia predecir
siempre la clase mayoritaria. SMOTE (Synthetic Minority Oversampling Technique)
genera muestras sintéticas de las clases minoritarias interpolando entre vecinos
reales en el espacio de features, sin duplicar registros existentes.

Se usa `SMOTE` de `imbalanced-learn` con estrategia `'not majority'` para
balancear solo las clases minoritarias sin subrepresentar la mayoritaria.

> **Instalación requerida:** `pip install imbalanced-learn`

In [ ]:
try:
    from imblearn.over_sampling import SMOTE
    SMOTE_DISPONIBLE = True
except ImportError:
    print('⚠ imbalanced-learn no instalado.')
    print('  Ejecutar: pip install imbalanced-learn')
    SMOTE_DISPONIBLE = False

In [ ]:
if SMOTE_DISPONIBLE:
    # k_neighbors=5 es el default — funciona bien con datasets de este tamaño
    # random_state fijo para reproducibilidad
    smote = SMOTE(sampling_strategy='not majority', random_state=42, k_neighbors=5)
    X_bal, y_bal = smote.fit_resample(X_enc, y_enc)

    print('=== Distribución después de SMOTE ===')
    conteo_bal = Counter(y_bal)
    for clase, n in sorted(conteo_bal.items()):
        nombre = le_target.classes_[clase]
        n_orig = conteo[clase]
        print(f'  {clase} ({nombre}): {n:,} ({n/len(y_bal)*100:.1f}%) — antes: {n_orig:,}')

    print(f'\nTotal muestras después de SMOTE: {len(X_bal):,}')
    print(f'Muestras sintéticas añadidas: {len(X_bal) - len(X_enc):,}')
else:
    # Si no hay SMOTE disponible, continuar con datos originales
    X_bal, y_bal = X_enc.values, y_enc
    print('Continuando con datos originales sin balanceo.')

In [ ]:
# Visualizar balance antes y después
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
nombres_clases = le_target.classes_
colores = ['#3498db', '#e67e22', '#c0392b']

# Antes
vals_antes = [conteo[i] for i in range(len(nombres_clases))]
axes[0].bar(nombres_clases, vals_antes, color=colores)
axes[0].set_title('Antes de SMOTE')
axes[0].set_ylabel('Cantidad de registros')
for i, v in enumerate(vals_antes):
    axes[0].text(i, v + 50, f'{v:,}', ha='center', fontsize=9)

# Después
if SMOTE_DISPONIBLE:
    conteo_bal_dict = Counter(y_bal)
    vals_despues = [conteo_bal_dict[i] for i in range(len(nombres_clases))]
    axes[1].bar(nombres_clases, vals_despues, color=colores)
    axes[1].set_title('Después de SMOTE')
    for i, v in enumerate(vals_despues):
        axes[1].text(i, v + 50, f'{v:,}', ha='center', fontsize=9)
else:
    axes[1].set_title('SMOTE no disponible')

for ax in axes:
    ax.set_xlabel('Categoría de severidad')
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=15, ha='right')

plt.suptitle('Balance de clases — efecto de SMOTE')
plt.tight_layout()
plt.savefig('balance_smote.png', dpi=150)
plt.show()

## 5. Exportación del dataset listo para modelado

In [ ]:
import numpy as np

# Reconstruir DataFrame con nombres de columnas
X_bal_df = pd.DataFrame(X_bal, columns=X_enc.columns)
y_bal_series = pd.Series(y_bal, name='CATEGORIA_SEVERIDAD_ENC')
y_bal_nombre = pd.Series(
    le_target.inverse_transform(y_bal), name='CATEGORIA_SEVERIDAD'
)

df_modelado = pd.concat([X_bal_df, y_bal_nombre, y_bal_series], axis=1)
df_modelado.to_csv('siniestros_modelado.csv', index=False, encoding='utf-8')

print(f'✓ Dataset para modelado exportado: {df_modelado.shape[0]:,} filas × {df_modelado.shape[1]} columnas')
print(f'\nArchivos generados en esta sesión:')
print('  siniestros_modelado.csv  — features encoded + SMOTE + target')
print('  balance_smote.png        — visualización de balanceo')
print('\nResumen de decisiones metodológicas:')
print(f'  Features incluidas:     {X_enc.shape[1]}')
print(f'  Features excluidas:     {len(excluir_real)} (ver tabla sección 3)')
print(f'  Registros originales:   {len(X_enc):,}')
print(f'  Registros tras SMOTE:   {len(X_bal):,}')
print(f'  Muestras sintéticas:    {len(X_bal) - len(X_enc):,}')
print('\nSiguiente paso: Modelado supervisado (Random Forest + XGBoost) — Notebook 03')

## Referencias metodológicas de este notebook

- **Bazarnovi & Mohammadian (2024)** — justificación de SMOTE para desbalance en datasets de accidentes de tránsito
- **Breiman (2001)** — Random Forest y compatibilidad con Label Encoding por splits numéricos
- **Provost & Fawcett (2013)** — criterios de selección de features por redundancia semántica y cardinalidad